# Per-window STM and magnitude log-likelihood

This notebook scores **each walk-forward window × realization separately**, conditioning on the **true observed history** up to the window start $t_0$, and evaluating the score on the **generated** events in $[t_0, t_0+H)$.

It is **not** the concatenated full-test-set STM used in `fine_slides_figures.ipynb`. That notebook sums over all windows into one catalog; here each window keeps its own score.

## What is loaded (no new forecasts)

For each cached horizon under the Hauksson pre-Ridgecrest rolling runs:

| File | Role |
|------|------|
| `analysis_cache/spacetime_loglikelihood.csv` | Per `(step_index, kind, method, seed)` space–time log-likelihood. For `kind==forecast`, history is observed events with $t < t_0$ and the scored points are that seed's synthetic catalog in the window. |
| `analysis_cache/magnitude_likelihoods.csv` | Per-event magnitude densities `fine_likelihood` / `etas_likelihood`. Built by walking windows with MAGNET FeatureState advanced by **observed** history only between windows. |

No FeatureState tips are read from disk. Magnitude densities are used as already cached.

## Metrics (per window $w$, seed $s$)

With $N_{w,s}$ = number of synthetic events in the window:

$$
\mathrm{LL}_{\mathrm{st}}(w,s) = \sum_{i \in w,s} \log \lambda(t_i,x_i,y_i) - \Lambda_w
$$

(already in the ST cache; $\lambda$ is the ETAS intensity from that window's inversion),

$$
\mathrm{LL}_{m}(w,s) = \sum_{e \in w,s} \log p^{(\mathrm{model})}(m_e)
$$

$$
\mathrm{LL}_{\mathrm{STM}}(w,s) = \mathrm{LL}_{\mathrm{st}}(w,s) + \mathrm{LL}_{m}(w,s),
\qquad
\overline{\mathrm{LL}}_{\mathrm{STM}}(w,s) = \mathrm{LL}_{\mathrm{STM}}(w,s)/N_{w,s}
$$

Figures plot **minus** the per-event mean, $-\overline{\mathrm{LL}}_{\mathrm{STM}}$, so lower is better (same convention as the slide deck). Empty windows ($N=0$) have undefined per-event means and are dropped from mean-based plots; the raw ST compensator-only score is kept in the table.

Observed-test reference per window uses `kind==observed_test` plus observed test magnitudes in the same time bounds.


In [ ]:
from __future__ import annotations

%matplotlib inline

import io
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# ---------------------------------------------------------------------------
# Style and paths
# ---------------------------------------------------------------------------
STYLE = {
    "font": {"family": "DejaVu Sans", "size": 12, "title": 13, "label": 12, "tick": 11, "legend": 10},
    "colors": {
        "methods": {"etas": "#284473", "FINE": "#E6225B"},
        "observed": "#1a1a1a",
        "grid": "#d0d0d0",
        "reference": "#808080",
    },
    "labels": {"etas": "ETAS", "FINE": "FINE"},
    "fill_alpha": 0.18,
    "line_width": 1.5,
    "display_dpi": 110,
    "save_dpi": 300,
    "save_png": True,
    "save_pdf": False,
    "save_svg": False,
    "overwrite": True,
}

METHODS = ["etas", "FINE"]
# Density column in magnitude_likelihoods.csv for each method.
MAG_DENSITY = {"etas": "etas_likelihood", "FINE": "fine_likelihood"}

# Horizons to analyse and plot. Comment out a line to drop that horizon.
# Set to None to use every cached horizon that has both ST and mag caches.
HORIZONS: list[float] | None = [
    0.5,
    1.0,
    # 4.0,
    7.0,
    60.0,
    300.0,
]

def find_repo_root() -> Path:
    for start in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (start / "runnable_code" / "presentation_figures.py").is_file():
            return start
    raise FileNotFoundError("Open this notebook from the etas checkout.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import etas.rolling_analysis as rolling_analysis

FIGURE_DIR = REPO_ROOT / "outputs" / "per_window_stm_ll"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": STYLE["font"]["family"],
    "font.size": STYLE["font"]["size"],
    "axes.titlesize": STYLE["font"]["title"],
    "axes.labelsize": STYLE["font"]["label"],
    "xtick.labelsize": STYLE["font"]["tick"],
    "ytick.labelsize": STYLE["font"]["tick"],
    "legend.fontsize": STYLE["font"]["legend"],
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def method_label(method: str) -> str:
    return STYLE["labels"].get(method, method)


def method_color(method: str) -> str:
    return STYLE["colors"]["methods"].get(method, "#333333")


def style_horizon_axis(ax: plt.Axes, horizons: list[float]) -> None:
    """Match fine_slides_figures x-axis: log scale, one tick per horizon ("7 d").

    Unlike presentation_figures._style_horizon_axis, do **not** draw vertical
    reference lines at each horizon.
    """
    from matplotlib.ticker import FixedFormatter, FixedLocator, NullLocator

    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator(horizons))
    ax.xaxis.set_major_formatter(FixedFormatter([f"{days:g} d" for days in horizons]))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlabel("Forecast horizon")


def horizon_tag(days: float) -> str:
    return f"horizon_{f'{days:g}'.replace('.', 'p')}d"


def export(fig, filename: str, *folder: str) -> None:
    directory = FIGURE_DIR.joinpath(*folder) if folder else FIGURE_DIR
    directory.mkdir(parents=True, exist_ok=True)
    if STYLE["save_png"]:
        fig.savefig(directory / f"{filename}.png", dpi=STYLE["save_dpi"], bbox_inches="tight")
    if STYLE["save_pdf"]:
        fig.savefig(directory / f"{filename}.pdf", dpi=STYLE["save_dpi"], bbox_inches="tight")
    if STYLE["save_svg"]:
        fig.savefig(directory / f"{filename}.svg", dpi=STYLE["save_dpi"], bbox_inches="tight")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=STYLE["display_dpi"], bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))


display(Markdown(f"**Repo:** `{REPO_ROOT}`  \n**Figures:** `{FIGURE_DIR}`"))


## Discover cached horizons

Looks for `spacetime_loglikelihood.csv` **and** `magnitude_likelihoods.csv` under the same experiment roots used by the slide notebook (short: 0.5/1/4 d; main: 7/60/300 d).


In [ ]:
EXPERIMENTS = [
    {
        "output_root": REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest_v3",
        "horizon_days": [0.5, 1.0, 4.0, 7.0, 60.0, 300.0],
    },
]

specs: list[dict] = []
for experiment in EXPERIMENTS:
    root = experiment["output_root"]
    if not root.is_dir():
        continue
    for days in experiment["horizon_days"]:
        horizon_dir = rolling_analysis.horizon_directory(root, float(days))
        cache = horizon_dir / "analysis_cache"
        st_path = cache / "spacetime_loglikelihood.csv"
        if not st_path.is_file():
            alt = horizon_dir / "spacetime_loglikelihood.csv"
            st_path = alt if alt.is_file() else st_path
        mag_path = cache / "magnitude_likelihoods.csv"
        if st_path.is_file() and mag_path.is_file():
            specs.append({
                "horizon_days": float(days),
                "output_root": root,
                "horizon_dir": horizon_dir,
                "spacetime_path": st_path,
                "magnitude_path": mag_path,
            })

if HORIZONS is not None:
    chosen = {float(d) for d in HORIZONS}
    skipped = [s for s in specs if s["horizon_days"] not in chosen]
    specs = [s for s in specs if s["horizon_days"] in chosen]
else:
    skipped = []
specs = sorted(specs, key=lambda s: s["horizon_days"])
HORIZON_LIST = [s["horizon_days"] for s in specs]

display(Markdown(
    "**Horizons analysed:** "
    + (", ".join(f"{d:g} d" for d in HORIZON_LIST) if HORIZON_LIST else "*none*")
    + (
        "  \n**Skipped (not in `HORIZONS`):** "
        + ", ".join(f"{s['horizon_days']:g} d" for s in skipped)
        if skipped
        else ""
    )
))
for s in specs:
    display(Markdown(
        f"- `{s['horizon_days']:g} d`: ST `{s['spacetime_path'].relative_to(REPO_ROOT)}`, "
        f"mag `{s['magnitude_path'].relative_to(REPO_ROOT)}`"
    ))


## Build per-window × seed scores

### Why other notebooks show finite mag LL, but this one had empty mag figures

`magnitude_likelihoods.csv` itself is fine. The slide notebook (`fine_slides_figures.ipynb`) never assigns events to windows: for each seed it takes **all** forecast rows and averages $\log p(m)$. That always works.

This notebook needs a **per-window** mag score, so it must join mag events into each ST window by time:

$$
t_0 \le \texttt{time_unix} < t_0 + H
$$

If that join returns **zero** events for a row with $N>0$:

- `mag_log_likelihood_mean` → **NaN** (empty mean) → mag figures empty
- old code set `mag_log_likelihood_sum = 0` → STM stayed finite but was **ST only**

So empty mag panels are not “bad densities in the cache”; they mean the window time-join failed (or the kernel reused a broken `per_window` table). Check the build cell’s `n_mag_joined` total and mismatch count before trusting plots.

### What is loaded

| File | Role |
|------|------|
| `spacetime_loglikelihood.csv` | Per `(step_index, method, seed)` ST LL for `kind==forecast` (history = observed $t<t_0$). |
| `magnitude_likelihoods.csv` | Per-event `fine_likelihood` / `etas_likelihood` (same file the slides use). |

### NaN policy

| Case | mag sum | mag mean | STM |
|------|---------|----------|-----|
| $N=0$ | 0 | NaN | ST (= $-\Lambda$); per-event mean NaN |
| $N>0$, join OK | $\sum\log p$ | mean | ST + mag |
| $N>0$, join fails | NaN | NaN | NaN + cell raises |


In [ ]:
def _sum_log(density: np.ndarray) -> float:
    values = np.log(np.asarray(density, dtype=float))
    values = values[np.isfinite(values)]
    return float(values.sum()) if values.size else float("nan")


def _mean_log(density: np.ndarray) -> float:
    values = np.log(np.asarray(density, dtype=float))
    values = values[np.isfinite(values)]
    return float(values.mean()) if values.size else float("nan")


def _to_unix_seconds(series: pd.Series) -> np.ndarray:
    """Datetime-like series -> unix seconds (int64), same units as time_unix."""
    values = pd.to_datetime(series, format="mixed")
    return (values.to_numpy(dtype="datetime64[ns]").astype("int64") // 10**9).astype(np.int64)


def _mag_window_totals(
    times: np.ndarray,
    density: np.ndarray,
    starts: np.ndarray,
    ends: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """For each window [start, end), sum/mean of log density and event count.

    ``times`` must be sorted ascending. Uses searchsorted — no Python event loop.
    """
    left = np.searchsorted(times, starts, side="left")
    right = np.searchsorted(times, ends, side="left")
    n = (right - left).astype(np.int64)
    # prefix sums of log density; log of non-positive -> -inf, dropped via nan handling below
    logp = np.log(np.asarray(density, dtype=float))
    logp = np.where(np.isfinite(logp), logp, 0.0)
    csum = np.concatenate([[0.0], np.cumsum(logp)])
    sums = csum[right] - csum[left]
    means = np.full(starts.shape, np.nan, dtype=float)
    positive = n > 0
    means[positive] = sums[positive] / n[positive].astype(float)
    sums = sums.astype(float)
    sums[~positive] = 0.0
    return n, sums, means


def per_window_scores(spacetime_path: Path, magnitude_path: Path, horizon_days: float) -> pd.DataFrame:
    """One row per (step_index, method, seed) forecast window, plus observed refs."""
    st = pd.read_csv(
        spacetime_path,
        usecols=[
            "step_index", "kind", "method", "seed",
            "window_start", "window_end", "n_events", "log_likelihood",
            "log_intensity_sum", "compensator",
        ],
    )
    mag = pd.read_csv(
        magnitude_path,
        usecols=[
            "population", "source", "seed", "time_unix",
            "fine_likelihood", "etas_likelihood",
        ],
    )

    forecast = st.loc[st["kind"] == "forecast"].copy()
    forecast["method"] = forecast["method"].astype(str)
    forecast["seed"] = forecast["seed"].astype(int)
    forecast["start_unix"] = _to_unix_seconds(forecast["window_start"])
    forecast["end_unix"] = _to_unix_seconds(forecast["window_end"])
    forecast["window_start"] = pd.to_datetime(forecast["window_start"], format="mixed")
    forecast["window_end"] = pd.to_datetime(forecast["window_end"], format="mixed")

    observed_st = st.loc[st["kind"] == "observed_test"].copy()
    observed_st["start_unix"] = _to_unix_seconds(observed_st["window_start"])
    observed_st["end_unix"] = _to_unix_seconds(observed_st["window_end"])
    obs_st_by_step = observed_st.set_index("step_index")

    observed_mag = mag.loc[
        (mag["population"] == "test") & (mag["source"] == "observed")
    ].copy()
    forecast_mag = mag.loc[mag["population"] == "forecast"].copy()
    forecast_mag["source"] = forecast_mag["source"].astype(str)
    forecast_mag["seed"] = forecast_mag["seed"].astype(int)

    if forecast_mag.empty:
        raise RuntimeError(
            f"No forecast rows in {magnitude_path}. "
            "Mag LL cannot be joined; rebuild magnitude_likelihoods.csv."
        )

    # Sorted mag series per (source, seed).
    mag_index: dict[tuple[str, int], tuple[np.ndarray, dict[str, np.ndarray]]] = {}
    for (source, seed), group in forecast_mag.groupby(["source", "seed"], sort=False):
        times = group["time_unix"].to_numpy(dtype=np.int64)
        order = np.argsort(times, kind="mergesort")
        times = times[order]
        dens = {
            "fine_likelihood": group["fine_likelihood"].to_numpy(dtype=float)[order],
            "etas_likelihood": group["etas_likelihood"].to_numpy(dtype=float)[order],
        }
        mag_index[(str(source), int(seed))] = (times, dens)

    obs_times = observed_mag["time_unix"].to_numpy(dtype=np.int64)
    obs_order = np.argsort(obs_times, kind="mergesort")
    obs_times = obs_times[obs_order]
    obs_dens = {
        "fine_likelihood": observed_mag["fine_likelihood"].to_numpy(dtype=float)[obs_order],
        "etas_likelihood": observed_mag["etas_likelihood"].to_numpy(dtype=float)[obs_order],
    }

    pieces: list[pd.DataFrame] = []
    for (method, seed), group in forecast.groupby(["method", "seed"], sort=False):
        method = str(method)
        seed = int(seed)
        column = MAG_DENSITY[method]
        group = group.sort_values("step_index").reset_index(drop=True)
        starts = group["start_unix"].to_numpy(dtype=np.int64)
        ends = group["end_unix"].to_numpy(dtype=np.int64)

        times, dens = mag_index.get((method, seed), (np.array([], dtype=np.int64), {}))
        if times.size == 0:
            n_mag = np.zeros(len(group), dtype=np.int64)
            mag_sum = np.zeros(len(group), dtype=float)
            mag_mean = np.full(len(group), np.nan, dtype=float)
        else:
            n_mag, mag_sum, mag_mean = _mag_window_totals(
                times, dens[column], starts, ends,
            )

        # Observed mag in the same windows (method-specific density column).
        obs_n, obs_mag_sum, obs_mag_mean = _mag_window_totals(
            obs_times, obs_dens[column], starts, ends,
        )

        out = group[[
            "step_index", "method", "seed", "window_start", "window_end",
            "n_events", "log_likelihood",
        ]].copy()
        out["horizon_days"] = float(horizon_days)
        out["n_mag_joined"] = n_mag
        out["st_log_likelihood"] = out["log_likelihood"].astype(float)
        out["mag_log_likelihood_sum"] = mag_sum
        out["mag_log_likelihood_mean"] = mag_mean

        n_events = out["n_events"].to_numpy(dtype=np.int64)
        # Non-empty ST window with zero joined mag events => join failure (NaN, not 0).
        failed = (n_events > 0) & (n_mag == 0)
        if failed.any():
            out.loc[failed, "mag_log_likelihood_sum"] = np.nan
            out.loc[failed, "mag_log_likelihood_mean"] = np.nan

        stm = out["st_log_likelihood"] + out["mag_log_likelihood_sum"]
        out["stm_log_likelihood"] = stm
        stm_mean = np.full(len(out), np.nan, dtype=float)
        positive_n = n_events > 0
        stm_mean[positive_n] = stm.to_numpy(dtype=float)[positive_n] / n_events[positive_n].astype(float)
        out["stm_mean_log_likelihood"] = stm_mean
        out["neg_stm_mean_log_likelihood"] = -out["stm_mean_log_likelihood"]
        out["neg_mag_mean_log_likelihood"] = -out["mag_log_likelihood_mean"]

        # Observed ST reference by step_index.
        obs_st_ll = obs_st_by_step.reindex(out["step_index"])["log_likelihood"].to_numpy(dtype=float)
        obs_n_st = obs_st_by_step.reindex(out["step_index"])["n_events"].to_numpy(dtype=float)
        obs_n_st = np.nan_to_num(obs_n_st, nan=0.0).astype(np.int64)
        obs_failed = (obs_n_st > 0) & (obs_n == 0)
        obs_mag_sum = obs_mag_sum.astype(float)
        obs_mag_mean = obs_mag_mean.astype(float)
        if obs_failed.any():
            obs_mag_sum[obs_failed] = np.nan
            obs_mag_mean[obs_failed] = np.nan

        out["observed_st_log_likelihood"] = obs_st_ll
        out["observed_n_events"] = obs_n_st
        out["observed_mag_log_likelihood_sum"] = obs_mag_sum
        out["observed_mag_log_likelihood_mean"] = obs_mag_mean
        obs_stm = obs_st_ll + obs_mag_sum
        out["observed_stm_log_likelihood"] = obs_stm
        obs_stm_mean = np.full(len(out), np.nan, dtype=float)
        obs_positive = obs_n_st > 0
        obs_stm_mean[obs_positive] = obs_stm[obs_positive] / obs_n_st[obs_positive].astype(float)
        out["observed_stm_mean_log_likelihood"] = obs_stm_mean
        out["neg_observed_stm_mean_log_likelihood"] = -out["observed_stm_mean_log_likelihood"]
        out["neg_observed_mag_mean_log_likelihood"] = -out["observed_mag_log_likelihood_mean"]
        pieces.append(out.drop(columns=["log_likelihood"]))

    return pd.concat(pieces, ignore_index=True)


parts = [
    per_window_scores(s["spacetime_path"], s["magnitude_path"], s["horizon_days"])
    for s in specs
]
per_window = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if len(per_window):
    mismatch = per_window.loc[per_window["n_events"] != per_window["n_mag_joined"]]
    empty = int((per_window["n_events"] == 0).sum())
    n_mag_total = int(per_window["n_mag_joined"].sum())
    finite_mag = int(np.isfinite(per_window["mag_log_likelihood_mean"]).sum())
    # Loud failure: this is exactly the state that produced empty mag figures before.
    if n_mag_total == 0 and int((per_window["n_events"] > 0).sum()) > 0:
        raise RuntimeError(
            "Magnitude time-join joined 0 events while ST has non-empty windows. "
            "Mag means would be all NaN and STM would silently equal ST. "
            f"Check magnitude_path / time_unix vs window_start. "
            f"Example magnitude file: {specs[0]['magnitude_path'] if specs else None}"
        )
    if len(mismatch):
        raise RuntimeError(
            f"ST vs mag event-count mismatch on {len(mismatch):,} / {len(per_window):,} rows. "
            f"n_mag_joined total={n_mag_total:,}."
        )
    display(Markdown(
        f"Built **{len(per_window):,}** window×seed rows across "
        f"**{per_window['horizon_days'].nunique()}** horizons, "
        f"**{per_window['method'].nunique()}** methods.\n\n"
        f"- Mag events joined: **{n_mag_total:,}** "
        f"(finite per-event mag means: **{finite_mag:,}**).\n"
        f"- ST vs mag mismatches: **{len(mismatch):,}** (must be 0).\n"
        f"- Empty synthetic windows (N=0): **{empty:,}** "
        f"(per-event means NaN by definition)."
    ))
    display(per_window.groupby(["horizon_days", "method"], sort=True).agg(
        n_rows=("seed", "size"),
        n_steps=("step_index", "nunique"),
        n_seeds=("seed", "nunique"),
        n_mag=("n_mag_joined", "sum"),
        mean_neg_stm=("neg_stm_mean_log_likelihood", "mean"),
        mean_neg_mag=("neg_mag_mean_log_likelihood", "mean"),
    ).reset_index())
else:
    display(Markdown("*No per-window rows. Check that both caches exist for at least one horizon.*"))


## Intermediate aggregate: mean over seeds within each window

For histograms and progress curves we first average the seed-wise scores inside each `(horizon, method, step_index)` window. The band on progress plots is the sample standard deviation across seeds.


In [ ]:
if len(per_window):
    window_mean = (
        per_window.groupby(["horizon_days", "method", "step_index"], sort=True)
        .agg(
            window_start=("window_start", "first"),
            window_end=("window_end", "first"),
            n_seeds=("seed", "nunique"),
            mean_n_events=("n_events", "mean"),
            stm_mean=("stm_mean_log_likelihood", "mean"),
            stm_std=("stm_mean_log_likelihood", "std"),
            neg_stm_mean=("neg_stm_mean_log_likelihood", "mean"),
            neg_stm_std=("neg_stm_mean_log_likelihood", "std"),
            mag_mean=("mag_log_likelihood_mean", "mean"),
            neg_mag_mean=("neg_mag_mean_log_likelihood", "mean"),
            neg_mag_std=("neg_mag_mean_log_likelihood", "std"),
            st_mean=("st_log_likelihood", "mean"),
            neg_obs_stm_mean=("neg_observed_stm_mean_log_likelihood", "first"),
            obs_stm_mean=("observed_stm_mean_log_likelihood", "first"),
            neg_obs_mag_mean=("neg_observed_mag_mean_log_likelihood", "first"),
        )
        .reset_index()
    )
    finite_mag_windows = int(np.isfinite(window_mean["neg_mag_mean"]).sum())
    if finite_mag_windows == 0:
        raise RuntimeError(
            "window_mean has zero finite neg_mag_mean values. "
            "Re-run the build cell; do not plot from a stale/broken per_window."
        )
    display(Markdown(
        f"Window-level table: **{len(window_mean):,}** rows. "
        f"Finite −mean mag LL windows: **{finite_mag_windows:,}**."
    ))
    display(window_mean.head(8))
else:
    window_mean = pd.DataFrame()


## Histograms — distribution of the per-window metric

Each panel is one horizon. The histogram is over **forecast windows**: each count is one window's seed-averaged $-\overline{\mathrm{LL}}_{\mathrm{STM}}$.

Dashed vertical lines mark the mean of the **observed-test** per-window $-\overline{\mathrm{LL}}_{\mathrm{STM}}$ (averaged over windows), under each method's magnitude density.


In [ ]:
def fig_metric_histograms(
    table: pd.DataFrame,
    *,
    value_col: str,
    observed_col: str,
    title: str,
    xlabel: str,
) -> plt.Figure | None:
    if table.empty:
        return None
    horizons = sorted(table["horizon_days"].unique())
    n = len(horizons)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows), squeeze=False)
    for ax, days in zip(axes.ravel(), horizons):
        sub = table.loc[np.isclose(table["horizon_days"], days)]
        for method in METHODS:
            vals = sub.loc[sub["method"] == method, value_col].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            ax.hist(
                vals,
                bins=min(30, max(8, int(np.sqrt(vals.size)))),
                density=True,
                alpha=0.45,
                color=method_color(method),
                label=f"{method_label(method)} (n={vals.size})",
            )
            obs = sub.loc[sub["method"] == method, observed_col].to_numpy(dtype=float)
            obs = obs[np.isfinite(obs)]
            if obs.size:
                ax.axvline(
                    float(np.mean(obs)),
                    color=method_color(method),
                    linestyle="--",
                    linewidth=1.2,
                    label=f"{method_label(method)} obs mean",
                )
        ax.set_title(f"T = {days:g} d")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("density")
        ax.grid(True, axis="y", color=STYLE["colors"]["grid"], linewidth=0.6)
        ax.legend(frameon=False, fontsize=8)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


if len(window_mean):
    export(
        fig_metric_histograms(
            window_mean,
            value_col="neg_stm_mean",
            observed_col="neg_obs_stm_mean",
            title="Per-window −mean STM LL (seed-averaged)",
            xlabel=r"$-\overline{\mathrm{LL}}_{\mathrm{STM}}$",
        ),
        "hist_neg_stm_mean_per_window",
        "histograms",
    )
    export(
        fig_metric_histograms(
            window_mean,
            value_col="neg_mag_mean",
            observed_col="neg_obs_mag_mean",
            title="Per-window −mean magnitude LL (seed-averaged)",
            xlabel=r"$-\overline{\log p(m)}$",
        ),
        "hist_neg_mag_mean_per_window",
        "histograms",
    )
else:
    display(Markdown("*No histograms: empty window table.*"))


## Progress — metric vs forecast window

For each horizon, plot seed-averaged $-\overline{\mathrm{LL}}_{\mathrm{STM}}$ against window start time. Shaded band = ±1 sample std across seeds. Dashed curves are the **observed-test** per-window score (same ST history/events; magnitude density is method-specific, so ETAS and FINE observed curves can differ slightly). Windows with $N=0$ (no events above $m_c$) yield NaN per-event means and appear as gaps.


In [ ]:
def fig_metric_progress(
    table: pd.DataFrame,
    *,
    value_col: str,
    std_col: str,
    observed_col: str,
    title: str,
    ylabel: str,
) -> list[tuple[float, plt.Figure]]:
    figures: list[tuple[float, plt.Figure]] = []
    if table.empty:
        return figures
    for days in sorted(table["horizon_days"].unique()):
        sub = table.loc[np.isclose(table["horizon_days"], days)].sort_values("window_start")
        fig, ax = plt.subplots(figsize=(10.5, 3.8))
        for method in METHODS:
            m = sub.loc[sub["method"] == method]
            if m.empty:
                continue
            x = pd.to_datetime(m["window_start"])
            y = m[value_col].to_numpy(dtype=float)
            s = m[std_col].to_numpy(dtype=float) if std_col in m.columns else np.zeros(len(m))
            s = np.nan_to_num(s, nan=0.0)
            color = method_color(method)
            ax.plot(x, y, color=color, lw=STYLE["line_width"], label=method_label(method))
            ax.fill_between(x, y - s, y + s, color=color, alpha=STYLE["fill_alpha"], linewidth=0)
            obs = m[observed_col].to_numpy(dtype=float)
            ax.plot(
                x, obs,
                color=color, linestyle="--", linewidth=1.0, alpha=0.85,
                label=f"{method_label(method)} observed",
            )
        ax.set_title(f"{title} — T = {days:g} d")
        ax.set_xlabel("forecast window start")
        ax.set_ylabel(ylabel)
        ax.grid(True, axis="y", color=STYLE["colors"]["grid"], linewidth=0.6)
        ax.legend(frameon=False, ncols=2)
        fig.autofmt_xdate()
        fig.tight_layout()
        figures.append((float(days), fig))
    return figures


if len(window_mean):
    for days, fig in fig_metric_progress(
        window_mean,
        value_col="neg_stm_mean",
        std_col="neg_stm_std",
        observed_col="neg_obs_stm_mean",
        title="Per-window −mean STM LL",
        ylabel=r"$-\overline{\mathrm{LL}}_{\mathrm{STM}}$",
    ):
        export(fig, "progress_neg_stm_mean", horizon_tag(days))

    for days, fig in fig_metric_progress(
        window_mean,
        value_col="neg_mag_mean",
        std_col="neg_mag_std",
        observed_col="neg_obs_mag_mean",
        title="Per-window −mean magnitude LL",
        ylabel=r"$-\overline{\log p(m)}$",
    ):
        export(fig, "progress_neg_mag_mean", horizon_tag(days))
else:
    display(Markdown("*No progress figures.*"))


## Raw (window × seed) histogram

Same metrics as the seed-averaged histograms above, but every realization is a separate sample (not seed-averaged first). Useful to see within-window spread. STM and magnitude −mean LL are both shown.


In [ ]:
if len(per_window):
    # Row-level table compatible with the hist helper (one sample per window×seed).
    raw = per_window.rename(columns={
        "neg_stm_mean_log_likelihood": "neg_stm_mean",
        "neg_observed_stm_mean_log_likelihood": "neg_obs_stm_mean",
        "neg_mag_mean_log_likelihood": "neg_mag_mean",
        "neg_observed_mag_mean_log_likelihood": "neg_obs_mag_mean",
    })
    export(
        fig_metric_histograms(
            raw,
            value_col="neg_stm_mean",
            observed_col="neg_obs_stm_mean",
            title="Per-window×seed −mean STM LL (all realizations)",
            xlabel=r"$-\overline{\mathrm{LL}}_{\mathrm{STM}}$",
        ),
        "hist_neg_stm_mean_window_seed",
        "histograms",
    )
    export(
        fig_metric_histograms(
            raw,
            value_col="neg_mag_mean",
            observed_col="neg_obs_mag_mean",
            title="Per-window×seed −mean mag LL (all realizations)",
            xlabel=r"$-\overline{\log p(m)}$",
        ),
        "hist_neg_mag_mean_window_seed",
        "histograms",
    )
else:
    display(Markdown("*Skipped raw histogram.*"))


## Summary across horizons

Mean ± std of the **seed-averaged per-window** scores, taken over windows:

1. $-\overline{\mathrm{LL}}_{\mathrm{STM}}$ (STM)
2. $-\overline{\log p(m)}$ (magnitude only)

This is a different aggregation than the concatenated-catalog STM in the slide notebook (there: sum over windows, then / total $N$; here: mean of per-window means).


In [ ]:
if len(window_mean):
    summary_rows = []
    for (days, method), g in window_mean.groupby(["horizon_days", "method"], sort=True):
        stm_vals = g["neg_stm_mean"].to_numpy(dtype=float)
        stm_vals = stm_vals[np.isfinite(stm_vals)]
        stm_obs = g["neg_obs_stm_mean"].to_numpy(dtype=float)
        stm_obs = stm_obs[np.isfinite(stm_obs)]
        mag_vals = g["neg_mag_mean"].to_numpy(dtype=float)
        mag_vals = mag_vals[np.isfinite(mag_vals)]
        mag_obs = g["neg_obs_mag_mean"].to_numpy(dtype=float)
        mag_obs = mag_obs[np.isfinite(mag_obs)]
        summary_rows.append({
            "horizon_days": float(days),
            "method": method,
            "n_windows": int(stm_vals.size),
            "mean_neg_stm_mean": float(stm_vals.mean()) if stm_vals.size else np.nan,
            "std_neg_stm_mean": float(stm_vals.std(ddof=1)) if stm_vals.size > 1 else 0.0,
            "mean_neg_obs_stm_mean": float(stm_obs.mean()) if stm_obs.size else np.nan,
            "n_windows_mag": int(mag_vals.size),
            "mean_neg_mag_mean": float(mag_vals.mean()) if mag_vals.size else np.nan,
            "std_neg_mag_mean": float(mag_vals.std(ddof=1)) if mag_vals.size > 1 else 0.0,
            "mean_neg_obs_mag_mean": float(mag_obs.mean()) if mag_obs.size else np.nan,
        })
    summary = pd.DataFrame(summary_rows)
    display(summary)

    def _summary_vs_horizon(
        *,
        value_col: str,
        std_col: str,
        observed_col: str,
        ylabel: str,
        title: str,
        filename: str,
    ) -> None:
        # Same layout language as fine_slides_figures.fig_loglikelihood_vs_horizon:
        # log x, ticks at each horizon labeled "7 d", shaded ±1 std, no vertical lines.
        fig, ax = plt.subplots(figsize=(7.2, 4.2))
        horizons = sorted(summary["horizon_days"].unique().tolist())
        style_horizon_axis(ax, horizons)
        for method in METHODS:
            m = summary.loc[summary["method"] == method].sort_values("horizon_days")
            if m.empty:
                continue
            x = m["horizon_days"].to_numpy(dtype=float)
            y = m[value_col].to_numpy(dtype=float)
            s = m[std_col].to_numpy(dtype=float)
            s = np.where(np.isfinite(s), s, 0.0)
            color = method_color(method)
            ax.fill_between(
                x, y - s, y + s,
                color=color, alpha=STYLE["fill_alpha"], linewidth=0, zorder=2,
            )
            ax.plot(
                x, y,
                color=color,
                marker="o",
                lw=STYLE["line_width"],
                label=method_label(method),
                zorder=3,
            )
            ax.plot(
                x, m[observed_col].to_numpy(dtype=float),
                color=color,
                linestyle="--",
                linewidth=max(STYLE["line_width"] - 0.4, 0.8),
                label=f"{method_label(method)} observed",
                zorder=4,
            )
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(frameon=False)
        fig.tight_layout()
        export(fig, filename, "across_horizons")

    _summary_vs_horizon(
        value_col="mean_neg_stm_mean",
        std_col="std_neg_stm_mean",
        observed_col="mean_neg_obs_stm_mean",
        ylabel=r"mean over windows of $-\overline{\mathrm{LL}}_{\mathrm{STM}}$",
        title="Per-window STM metric vs horizon",
        filename="summary_neg_stm_mean_vs_horizon",
    )
    _summary_vs_horizon(
        value_col="mean_neg_mag_mean",
        std_col="std_neg_mag_mean",
        observed_col="mean_neg_obs_mag_mean",
        ylabel=r"mean over windows of $-\overline{\log p(m)}$",
        title="Per-window magnitude −mLL vs horizon",
        filename="summary_neg_mag_mean_vs_horizon",
    )

    # Persist tables for later reuse / verification.
    out_csv = FIGURE_DIR / "per_window_stm_scores.csv"
    per_window.to_csv(out_csv, index=False)
    window_mean.to_csv(FIGURE_DIR / "per_window_stm_seed_means.csv", index=False)
    summary.to_csv(FIGURE_DIR / "per_window_stm_horizon_summary.csv", index=False)
    display(Markdown(
        f"Wrote:\n- `{out_csv.relative_to(REPO_ROOT)}`\n"
        f"- `{(FIGURE_DIR / 'per_window_stm_seed_means.csv').relative_to(REPO_ROOT)}`\n"
        f"- `{(FIGURE_DIR / 'per_window_stm_horizon_summary.csv').relative_to(REPO_ROOT)}`"
    ))
else:
    display(Markdown("*No summary.*"))


## Per-horizon STM bars (FINE vs ETAS)

One bar chart per forecast horizon: seed-averaged per-window mean $-\overline{\mathrm{LL}}_{\mathrm{STM}}$, then averaged over windows (same summary values as the vs-horizon curves). Style matches the slide deck method bars — FINE (magenta) and ETAS (navy), y-label $\mathcal{L}$, no grid.


In [ ]:
def fig_stm_method_bars(
    values: dict[str, float],
    *,
    title: str,
    ylabel: str = r"$\mathcal{L}$",
    order: list[str] | None = None,
) -> plt.Figure | None:
    """Two-method bar chart in the slide style (no grid, bottom/left spines only)."""
    order = order or ["FINE", "etas"]
    chosen = [m for m in order if np.isfinite(values.get(m, np.nan))]
    if not chosen:
        return None
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    heights = np.array([float(values[m]) for m in chosen], dtype=float)
    ax.bar(
        np.arange(len(chosen)),
        heights,
        width=0.65,
        color=[method_color(m) for m in chosen],
        edgecolor="none",
    )
    ax.set_xticks(np.arange(len(chosen)), [method_label(m) for m in chosen])
    ax.set_ylabel(ylabel, fontsize=STYLE["font"]["label"] + 4)
    ax.set_title(title)
    # Start at 0 like the reference figure; pad the top slightly.
    ymax = float(np.nanmax(heights))
    ax.set_ylim(0.0, ymax * 1.12 if ymax > 0 else 1.0)
    ax.grid(False)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    return fig


if "summary" in globals() and len(summary):
    for days in HORIZON_LIST:
        sub = summary.loc[np.isclose(summary["horizon_days"], days)]
        if sub.empty:
            continue
        values = {
            str(row["method"]): float(row["mean_neg_stm_mean"])
            for _, row in sub.iterrows()
        }
        export(
            fig_stm_method_bars(
                values,
                title=f"Per-window STM — T = {days:g} d",
            ),
            "stm_bars_fine_etas",
            horizon_tag(days),
        )
else:
    display(Markdown("*No summary table for STM bars. Run the summary cell first.*"))


## Verification checklist

1. **Conditioning:** ST rows with `kind==forecast` were scored with observed history $t < t_0$ (see `etas/catalog_loglikelihood.py`). Mag densities were built the same way (`set_likelihood_history` / `extend_likelihood_history` with observed events only between windows).
2. **Unit:** one score per `(horizon, step_index, method, seed)` — not a concatenated catalog.
3. **STM:** `stm_log_likelihood = st_log_likelihood + mag_log_likelihood_sum`; per-event mean divides by `n_events` from the ST table.
4. **Join:** `n_events` should equal `n_mag_joined` (reported above).
5. **Sign:** stored columns include both positive LL and `neg_*` for plots; figures use minus mean LL.
6. **Contrast with slides:** `fine_slides_figures.ipynb` STM section sums $\mathrm{LL}_{\mathrm{st}}$ and $\log p(m)$ over **all** windows then divides by total $N$. Here each window is scored and plotted on its own.
